# 25 — Multi-Class Attack Classification on **Real AWS Traffic**

**Dijalankan di SageMaker.** Melengkapi notebook 24 (multi-kelas in-domain) dengan
multi-kelas pada trafik nyata AWS. Karena fase serangan AWS dijalankan **satu jenis
per selang waktu** (berurutan + cooldown), label *kategori* dapat diturunkan dari
timeline (bukan hanya biner):

Timeline serangan (selaras `attack_scenario.sh` varian `clean`, ~7 menit):

| menit | fase | kategori |
|---|---|---|
| 0–1 | benign warm-up | Benign |
| 1–3 | SSH brute-force | BruteForce |
| 3–5 | Slowloris | DoS |
| 5–6 | SYN flood | DDoS |
| 6–7 | benign cool-down | Benign |

**Notebook ini MEMERIKSA SENDIRI ketersediaan sumber** (tidak menebak), urut prioritas:
1. **CSV dgn kolom `gt_category`** (dari `unsw_extract_infer.py` mode detect ter-update) -> PALING AKURAT, pakai langsung;
2. pcap AWS mentah -> re-ekstrak 9 fitur + `elapsed` (dari first_seen) + kategori timeline;
3. CSV dgn `elapsed_sec` -> hitung kategori dari timeline; atau `ground_truth` teks;
4. bila CSV **hanya biner** (`ground_truth` 0/1 tanpa `gt_category`/`elapsed_sec`) -> multi-kelas TIDAK bisa; beri pesan.

Model multi-kelas diambil dari notebook 24 (dilatih pada kategori CIC). Evaluasi
hanya pada kategori yang beririsan: Benign, BruteForce, DoS, DDoS.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('xgboost','scikit-learn','pandas','numpy','matplotlib','boto3') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
# nfstream hanya perlu bila re-ekstrak dari pcap; coba pasang, tak fatal bila gagal
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
try:
    import nfstream  # noqa
    HAS_NFSTREAM=True
except Exception:
    HAS_NFSTREAM=False
print('setup ok; nfstream=',HAS_NFSTREAM)
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef, f1_score, balanced_accuracy_score
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='multiclass_aws_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
# kategori target yg beririsan dgn model CIC multi-kelas
AWS_CLASSES=['Benign','BruteForce','DoS','DDoS']
# Timeline ground-truth (menit) -> kategori. SELARAS dgn attack_scenario.sh + GT_CAT di
# unsw_extract_infer.py: SSH brute 1-3, Slowloris(DoS) 3-5, SYN flood(DDoS) 5-6, benign sisanya.
# (Dipakai HANYA bila CSV tak punya kolom gt_category / saat re-ekstrak dari pcap.)
GT=[(0,1,'Benign'),(1,3,'BruteForce'),(3,5,'DoS'),(5,6,'DDoS'),(6,7,'Benign')]
def gt_cat(elapsed_sec):
    m=elapsed_sec/60.0
    for a,z,lab in GT:
        if a<=m<z: return lab
    return 'Benign'
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'classes':AWS_CLASSES}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def find(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Deteksi sumber data AWS berlabel-kategori (periksa sendiri)

In [ ]:
PCAP=find(['/opt/ids2018/*.pcap','../data/*.pcap','./*.pcap','/opt/unsw/*.pcap','detect_*.pcap'])
# PENTING: pilih CSV yang BENAR-BENAR punya kolom kategori (gt_category), bukan sekadar *_flows.csv
# pertama (folder bisa berisi far_/d2_ benign yg hanya biner). Prioritas: gt_category > elapsed > ground_truth teks.
import glob as _glob
def _pick_cat_csv():
    cands=[]
    for pat in ['detect_*_flows.csv','./detect_*_flows.csv','../*_flows.csv','*_flows.csv','../data/*_flows.csv']:
        cands+=sorted(_glob.glob(pat))
    cands=list(dict.fromkeys(cands))
    best=None; best_rank=99
    for p in cands:
        try: cols=list(pd.read_csv(p,nrows=1).columns)
        except Exception: continue
        rank=99
        if 'gt_category' in cols: rank=0
        elif 'elapsed_sec' in cols or 'elapsed' in cols: rank=1
        elif 'ground_truth' in cols: rank=2
        if rank<best_rank: best,best_rank=p,rank
        if best_rank==0: break
    return best
CSV_CAT=_pick_cat_csv()
print('pcap ditemukan  :', PCAP)
print('csv terpilih    :', CSV_CAT)
source=None
if CSV_CAT:
    _h=pd.read_csv(CSV_CAT, nrows=3)
    print('  kolom CSV     :', [c for c in ['gt_category','elapsed_sec','ground_truth'] if c in _h.columns])
    if any(c in _h.columns for c in ['gt_category','elapsed_sec','elapsed','ground_truth','category','cat']): source='csv_cat'
elif PCAP and HAS_NFSTREAM: source='pcap'
print('SUMBER dipakai  :', source)
if source is None:
    print('\n[!] Tidak ada sumber berlabel-kategori (pcap + nfstream, atau csv dgn elapsed/ground_truth).')
    print('    `aws_labeled_all.csv` hanya biner (y 0/1) -> multi-kelas AWS TIDAK bisa dari situ.')
    print('    Opsi: sediakan pcap AWS + nfstream, ATAU regenerasi csv dgn kolom elapsed_sec.')
print('=== SEL 2 (deteksi sumber) SELESAI ===')

## 3. Bangun tabel (9 fitur + kategori) dari sumber terpilih

In [ ]:
def from_pcap(pcap):
    from nfstream import NFStreamer
    df=NFStreamer(source=pcap, statistical_analysis=True).to_pandas()
    dur_us=(df['bidirectional_duration_ms']*1000.0).replace(0,np.nan)
    dur_s=(df['bidirectional_duration_ms']/1000.0).replace(0,np.nan)
    out=pd.DataFrame({
        'duration':(df['bidirectional_duration_ms']*1000.0),
        'fwd_pkts':df['src2dst_packets'],'bwd_pkts':df['dst2src_packets'],
        'fwd_bytes':df['src2dst_bytes'],'bwd_bytes':df['dst2src_bytes'],
        'fwd_mean':df.get('src2dst_mean_ps',df['src2dst_bytes']/df['src2dst_packets'].replace(0,np.nan)),
        'bwd_mean':df.get('dst2src_mean_ps',df['dst2src_bytes']/df['dst2src_packets'].replace(0,np.nan)),
        'src_load':(df['src2dst_bytes']/dur_s),'dst_load':(df['dst2src_packets']/dur_s)})
    out=out.replace([np.inf,-np.inf],np.nan).fillna(0.0)
    fs=df['bidirectional_first_seen_ms']; elapsed=((fs-fs.min())/1000.0).values
    out['cat']=[gt_cat(e) for e in elapsed]
    return out

def from_csv_cat(p):
    d=pd.read_csv(p)
    ecol=next((c for c in ['elapsed_sec','elapsed'] if c in d.columns),None)
    if 'gt_category' in d.columns:
        # PALING AKURAT: kategori dari skrip unsw_extract_infer.py (mode detect ter-update),
        # diturunkan dari elapsed=first_seen (waktu MULAI flow), bukan urutan baris.
        d['cat']=d['gt_category'].astype(str)
    elif 'ground_truth' in d.columns and d['ground_truth'].dtype==object:
        d['cat']=d['ground_truth'].map(lambda s: s if s in AWS_CLASSES else ('BruteForce' if 'brute' in str(s).lower() else ('DDoS' if 'ddos' in str(s).lower() else ('DoS' if 'dos' in str(s).lower() else 'Benign'))))
    elif ecol is not None:
        d['cat']=[gt_cat(e) for e in d[ecol].values]
    else:
        return None
    miss=[c for c in CANON if c not in d.columns]
    if miss: print('csv kekurangan fitur:',miss); return None
    return d[CANON+['cat']]

aws=None
if source=='pcap': aws=from_pcap(PCAP)
elif source=='csv_cat': aws=from_csv_cat(CSV_CAT)
if aws is not None:
    aws=aws[aws['cat'].isin(AWS_CLASSES)].replace([np.inf,-np.inf],np.nan).dropna()
    print('AWS berlabel-kategori:',len(aws),'| distribusi:',aws['cat'].value_counts().to_dict())
else:
    print('(lewati) tak ada tabel berlabel-kategori; sel berikut akan no-op.')
print('=== SEL 3 (bangun tabel) SELESAI ===')

## 4. Muat model multi-kelas (notebook 24) + skor AWS

In [ ]:
# Kita latih ulang model multi-kelas CIC ringkas di sini (agar mandiri), lalu petakan
# prediksi ke kelas yg beririsan dgn AWS. Alternatif: muat model tersimpan dari nb24.
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, LabelEncoder
def load_cic_multiclass():
    p=find(['../../CICDDoS2018/data/file_100.csv'])
    if not p: return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    def mp(s):
        s=str(s).strip().lower()
        if s in ('benign','normal'): return 'Benign'
        if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
        if s.startswith('dos'): return 'DoS'
        if 'brute' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
        return 'Other'
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON}); d['cat']=c[LAB].map(mp)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat'].isin(AWS_CLASSES)]
    return d

clf=None; scaler=None; le=None
if aws is not None:
    cic=load_cic_multiclass()
    if cic is not None and cic['cat'].nunique()>=2:
        le=LabelEncoder().fit(AWS_CLASSES)
        scaler=StandardScaler().fit(cic[CANON].values)  # fit pada CIC-train
        Xtr=scaler.transform(cic[CANON].values); ytr=le.transform(cic['cat'].values)
        clf=xgb.XGBClassifier(objective='multi:softprob',num_class=len(AWS_CLASSES),max_depth=8,learning_rate=0.1,
            n_estimators=200,subsample=0.8,colsample_bytree=0.8,tree_method='hist',eval_metric='mlogloss',random_state=42,n_jobs=-1)
        clf.fit(Xtr,ytr); print('model multi-kelas CIC dilatih pada kelas:',list(le.classes_))
    else:
        print('CIC multi-kelas tak tersedia untuk melatih model.')
print('=== SEL 4 (model multi-kelas) SELESAI ===')

## 5. Zero-shot multi-kelas di AWS: confusion matrix + recall per-kategori

In [ ]:
def eval_mc(Xz,y_true_lab,tag):
    yp=le.inverse_transform(clf.predict(Xz))
    labels=AWS_CLASSES
    cm=confusion_matrix(y_true_lab,yp,labels=labels)
    rep=classification_report(y_true_lab,yp,labels=labels,target_names=labels,output_dict=True,zero_division=0)
    res={'tag':tag,'classes':labels,'n':int(len(y_true_lab)),
         'macro_f1':round(float(f1_score(y_true_lab,yp,labels=labels,average='macro',zero_division=0)),4),
         'balanced_acc':round(float(balanced_accuracy_score(y_true_lab,yp)),4),
         'per_class':{c:{'recall':round(rep[c]['recall'],4),'precision':round(rep[c]['precision'],4),
                         'support':int(rep[c]['support'])} for c in labels},
         'confusion':cm.tolist()}
    print(f'[{tag}] macro-F1={res["macro_f1"]} bal-acc={res["balanced_acc"]}')
    return res,cm,labels

def plot_cm(cm,labels,title,fname):
    M=np.array(cm,float); row=M.sum(1,keepdims=True); row[row==0]=1; M=M/row
    fig,ax=plt.subplots(figsize=(1.3+0.8*len(labels),1.1+0.8*len(labels)))
    im=ax.imshow(M,cmap='Blues',vmin=0,vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]): ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=8,color='white' if M[i,j]>0.5 else 'black')
    ax.set_title(title); fig.colorbar(im,ax=ax,fraction=0.046); plt.tight_layout(); savefig(fname)

RESULTS['zero_shot']=None
if aws is not None and clf is not None:
    Xz=scaler.transform(aws[CANON].values)
    res,cm,labels=eval_mc(Xz,aws['cat'].values,'zero-shot (CIC->AWS)')
    RESULTS['zero_shot']=res
    plot_cm(cm,labels,'AWS multi-class zero-shot (recall)','confusion_aws_mc_zeroshot.png')
    import IPython.display as ipd; ipd.display(pd.DataFrame(res['per_class']).T)
else:
    print('(lewati) prasyarat zero-shot tak lengkap.')
print('=== SEL 5 (zero-shot multi-kelas AWS) SELESAI ===')

## 6. Few-shot multi-kelas: tambah k% label AWS (per kategori) lalu skor ulang

In [ ]:
from sklearn.model_selection import train_test_split
RESULTS['few_shot']=[]
if aws is not None and clf is not None and aws['cat'].nunique()>=2:
    # split AWS: train pool (utk few-shot) + test tetap
    a_tr,a_te=train_test_split(aws,test_size=0.4,random_state=42,stratify=aws['cat'])
    cic=load_cic_multiclass()
    yte=a_te['cat'].values
    for k in [0,1,5,10,20]:
        if k==0:
            sc=StandardScaler().fit(cic[CANON].values)
            Xtr=sc.transform(cic[CANON].values); ytr=le.transform(cic['cat'].values)
        else:
            n=max(len(AWS_CLASSES),int(len(a_tr)*k/100))
            add=a_tr.groupby('cat',group_keys=False).apply(lambda g: g.sample(min(len(g),max(1,n//aws['cat'].nunique())),random_state=42))
            mix=pd.concat([cic[CANON+['cat']],add[CANON+['cat']]],ignore_index=True)
            sc=StandardScaler().fit(mix[CANON].values)
            Xtr=sc.transform(mix[CANON].values); ytr=le.transform(mix['cat'].values)
        m=xgb.XGBClassifier(objective='multi:softprob',num_class=len(AWS_CLASSES),max_depth=8,learning_rate=0.1,
            n_estimators=200,subsample=0.8,colsample_bytree=0.8,tree_method='hist',eval_metric='mlogloss',random_state=42,n_jobs=-1)
        m.fit(Xtr,ytr)
        yp=le.inverse_transform(m.predict(sc.transform(a_te[CANON].values)))
        mf=round(float(f1_score(yte,yp,labels=AWS_CLASSES,average='macro',zero_division=0)),4)
        ba=round(float(balanced_accuracy_score(yte,yp)),4)
        RESULTS['few_shot'].append({'k_percent':k,'macro_f1':mf,'balanced_acc':ba})
        print(f'k={k}% macro-F1={mf} bal-acc={ba}')
    import IPython.display as ipd; ipd.display(pd.DataFrame(RESULTS['few_shot']))
else:
    print('(lewati) prasyarat few-shot tak lengkap.')
print('=== SEL 6 (few-shot multi-kelas AWS) SELESAI ===')

## 7. Simpan + UPLOAD S3

In [ ]:
jp=os.path.join(OUTDIR,'multiclass_aws_results.json')
json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
if RESULTS.get('zero_shot') is None:
    print('\nCATATAN: multi-kelas AWS TIDAK dihasilkan (sumber berlabel-kategori tak tersedia).')
    print('Sediakan pcap AWS + nfstream di SageMaker, atau csv dgn kolom elapsed_sec/ground_truth.')
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/multiclass_aws/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/multiclass_aws/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 7 (simpan + upload) SELESAI ===')
print('SEMUA SELESAI.')